In [ ]:
#enable NEE for the notebook
%%configure 
{ 
   "conf": {
       "spark.native.enabled": "true", 
   } 
}

In [ ]:
# enable NEE at cell level
spark.conf.set('spark.native.enabled', 'true')

# disable NEE at cell level
spark.conf.set('spark.native.enabled', 'false')



In [ ]:
# Test 01
# This gets set of records loaded from sales folder that contains over 1.6 billion records, to raw object.
# Run this in default ENV, you will not see any NEE operations. It takes 30-40 seconds in my ENV.
# Run this in NEE enabled Environment, you will see NEE operations such as NativeFileScan, Velox.... It takes 8-10 seconds in my ENV.

sales_df = spark.read.parquet("Files/sales.parquet")
sales_df.createOrReplaceTempView("sales")
result = spark.sql("select count(*) from sales where product_id > 598794")
result.collect()
result.explain()

In [ ]:
# Test 02
# This loads data from two folders and combine, sales contains 1.6 billion records and product contains 5 million records.
# Then we perform some aggregation, grouping and union operations.
# Run this in default ENV, you will not see any NEE operations. It takes 150-180 seconds in my ENV.
# Run this in NEE enabled Environment, expand Spark Jobs details, see jobs, you will see velox related jobs. It takes 60-80 seconds in my ENV.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, lit, broadcast
import time

# loading data to dataframes
sales_df = spark.read.parquet("Files/sales.parquet")
product_df = spark.read.parquet("Files/products.parquet")

start_time = time.time()

# Broadcast the smaller DataFrame (products) for optimized join though it has 5M
product_df = broadcast(product_df)

# creating one set with join, aggrate and group
agg_with_brand = (
    sales_df
    .join(product_df, sales_df.product_id == product_df.product_id, "inner")
    .groupby("category", "brand")
    .agg(
        sum("quantity").alias("total_quantity"),
        avg("amount").alias("avg_amount"),
        count("*").alias("transaction_count")
    )
)

# creating the second set
agg_without_brand = (
    sales_df
    .join(product_df, sales_df.product_id == product_df.product_id, "inner")
    .groupby("category")
    .agg(
        sum("quantity").alias("total_quantity"),
        avg("amount").alias("avg_amount"),
        count("*").alias("transaction_count")
    )
    .withColumn("brand", lit(None)) 
)

# set the column order of the second set
agg_without_brand = agg_without_brand.select("category", "brand", "total_quantity", "avg_amount", "transaction_count")

# combine both
result_df = (
    agg_with_brand
    .union(agg_without_brand)
    .orderBy(col("category").asc(), col("brand").asc_nulls_last())  # Ensure NULLs appear last
)

# show results
result_df.show()

end_time = time.time()
print(f"Execution Time: {end_time - start_time:.2f} seconds")


In [ ]:
result_df.explain()